In [1]:
import requests
import json 
import ta
import tensorflow as tf
import keras
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
from matplotlib import pyplot as plt
import time
import datetime
from datetime import timezone
import urllib
from datetime import timedelta
import h5py
from urllib import parse
import pprint
import dateutil
import concurrent.futures as fut
import numpy as np
from keras import layers
import pandas as pd
import mplfinance
import csv
import ccxt
import bisect

In [ ]:
ki=ccxt.binance()
ku=ccxt.kucoin()

end1=datetime.datetime.now(datetime.timezone.utc)-datetime.timedelta(minutes=6)
#end1=datetime.datetime(2023,4,1,10,59,59)

end=datetime.datetime(end1.year,end1.month,end1.day-1,7,59,59,99)
starts=datetime.datetime(2018,1,1,0,0,0,0,timezone.utc)

print(starts)
timeframe=['1w','4h','15m']
def parsbi(start_,end_,fr):
    cans=[]
    since=start_
    while since<=end_:
        
        can=ki.fetch_ohlcv("BTC/USDT",fr,since,limit=1000)
        if not can:
            break
        c1=[c for c in can if c[0]<end_]
        if not c1:
            break
        cans.extend(c1)
        since = cans[-1][0] + 1  # Передвигаем `since` на последнюю свечу
        time.sleep(0.3) 
    return cans

data={}
def daa1(fr,start):

        start_ku=ki.parse8601(start.isoformat())
        end_ku=ki.parse8601(end.isoformat())

        return fr,parsbi(start_ku,end_ku,fr)


with fut.ThreadPoolExecutor(3) as tr:
                future=[tr.submit(daa1,fr,starts) for fr in timeframe]  
                for k in fut.as_completed(future):
                        fr,u=k.result()
                        
                        
                        data[fr]=u
with open("D:\Python1\AI\dataset\\"+"BTCUSDT_new"+'.json','w') as f:
                                
                                json.dump(data,f)

In [ ]:
path15=r'D:\Python1\AI\cripto_bot\from_3_3.parquet'
path4=r"D:\Python1\AI\cripto_bot\from_2_3.parquet"
path1=r"D:\Python1\AI\cripto_bot\from_1_3.parquet"




In [2]:
ki=ccxt.binance()
ku=ccxt.kucoin()
path15=r'D:\Python1\AI\cripto_bot\from_3_3.parquet'
path4=r"D:\Python1\AI\cripto_bot\from_2_3.parquet"
path1=r"D:\Python1\AI\cripto_bot\from_1_3.parquet"


class parse():
    def __init__(self):
        self.data15=pd.read_parquet(path15,engine='pyarrow')
        self.data4=pd.read_parquet(path4,engine='pyarrow')
        self.data1=pd.read_parquet(path1,engine='pyarrow')
        self.timeframe=['1w','4h','15m']
        print('11')

    def __call__(self,t):
        data={}
        #if t-self.data15['timestamp'].iloc[-1]>=timedelta(minutes=15):
        if t-datetime.datetime.fromisoformat("2019-03-12 00:00:00+00:00")>=timedelta(minutes=15):
            start=(self.data15['timestamp'].iloc[-1]+datetime.timedelta(minutes=15))
            start=datetime.datetime.fromisoformat("2019-03-12 00:00:00+00:00")
            end=t
            
            with fut.ThreadPoolExecutor(1) as tr:
                            #future=[tr.submit(self.daa1,fr,start,end) for fr in self.timeframe]
                            future=[tr.submit(self.daa1,self.timeframe[2],start,end),]  
                            for k in fut.as_completed(future):
                                    fr,u=k.result()
                                    data[fr]=u
            return data
    def parsbi(self,start_,end_,fr):
        cans=[]
        since=start_
        while since<=end_:
            
            can=ki.fetch_ohlcv("BTC/USDT",fr,since,limit=1000)
            if not can:
                break
            c1=[c for c in can if c[0]<end_]
            if not c1:
                break
            cans.extend(c1)
            since = cans[-1][0] + 1  # Передвигаем `since` на последнюю свечу
            time.sleep(0.1) 
        return cans
    
    def daa1(self,fr,start,end):
        if fr=='15m':
             start1=start - timedelta(minutes=15*100)
        elif fr=='4h':
             start1=start-timedelta(hours=4*100)
        elif fr=='1w':
             start1=start - timedelta(days=7*100)
        else:
             raise Exception('dont correct ft in daa1')
        
        start_ku=ki.parse8601(start1.isoformat())
        end_ku=ki.parse8601(end.isoformat())
        u=self.parsbi(start_ku,end_ku,fr)
        u=self.test_data(fr,u,start,end)
        for i in u:
             print(f'2:{datetime.datetime.fromtimestamp(i[0]/1000,timezone.utc)}')
        return fr,u
    def parsku(self,start_,end_,timeframe):
        cans=[]
        since=start_
        while since<=end_:
            can=ku.fetch_ohlcv("BTC/USDT",timeframe,since,limit=1000)
            if not can:
                break
            c1=[c for c in can if c[0]<end_]
            if not c1:
                break
            cans.extend(c1)
            since = cans[-1][0] + 1  # Передвигаем `since` на последнюю свечу 
        return cans

    def test_data(self,timeframe,data,start,end):
        m15=data
        dot=[]
        last=datetime.datetime.fromtimestamp(m15[0][0]/1000,datetime.timezone.utc)
        print(str('len m15:')+str(len(m15)))
        for ind,i in enumerate(m15):
        #if 10<ind<20:
         #    print(datetime.datetime.fromtimestamp(m15[ind][0]/1000,timezone.utc))
            if datetime.datetime.fromtimestamp(i[0]/1000,timezone.utc) < start:
                last=datetime.datetime.fromtimestamp(i[0]/1000,datetime.timezone.utc)
                continue
            if ind ==0:
                continue
            if datetime.datetime.fromtimestamp(i[0]/1000,datetime.timezone.utc)-datetime.timedelta(minutes=15)!=last:
                print('--------------------------')
                print(f'last0:{last}')
                start=last
                print(f'start1:{start.isoformat()}')
                end=datetime.datetime.fromtimestamp(i[0]/1000,datetime.timezone.utc)-datetime.timedelta(seconds=1)
                print(f'end1:{end.isoformat()}')
                print(f"i:{datetime.datetime.fromtimestamp(i[0]/1000,datetime.timezone.utc)}| last:{last}")

                start_ku_v=ku.parse8601((start-datetime.timedelta(minutes=15*99)).isoformat())
                end_ku_v=ku.parse8601(end.isoformat())
                #print('start ku:',(last-datetime.timedelta(hours=4*175)))
                #print('end ku:',last)
                #print(f'time:{datetime.datetime.fromtimestamp(m15[ind][0]/1000,timezone.utc)} | ind:{ind} | last time:{datetime.datetime.fromtimestamp(m15[ind-1][0]/1000,timezone.utc)} | time:{last}')
                #print('start bi:',datetime.datetime.fromtimestamp( dat[-175][0]/1000,timezone.utc))
                #print('end bi:',datetime.datetime.fromtimestamp(dat[-1][0]/1000,timezone.utc))
                ku_can_v=self.parsku(start_ku_v,end_ku_v,timeframe)

                start1=datetime.datetime.fromtimestamp(ku_can_v[0][0]/1000,timezone.utc)
                

                for o in ku_can_v:
                    if datetime.datetime.fromtimestamp(o[0]/1000,timezone.utc)!=start1:
                        print(f"ERROR({datetime.datetime.fromtimestamp(o[0]/1000,timezone.utc)} : {start1})")
                        start1=datetime.datetime.fromtimestamp(o[0]/1000,timezone.utc)
                    start1+=datetime.timedelta(minutes=15)
                mean_v1=np.array(data[:100])
                pl=datetime.datetime.fromtimestamp(mean_v1[-1][0]/1000,timezone.utc)
                for o in range(1,len(mean_v1)):
                    o+=1
                    o*=-1
                    if pl-datetime.timedelta(minutes=15)!=datetime.datetime.fromtimestamp(mean_v1[o][0]/1000,timezone.utc):
                        print("AAAA:",o,'|',datetime.datetime.fromtimestamp(mean_v1[o][0]/1000,timezone.utc))

                    pl=datetime.datetime.fromtimestamp(mean_v1[o][0]/1000,timezone.utc)
                mean_v1=mean_v1[:,-1]
                mean_v2=np.array(ku_can_v)[:100,-1]
                
                
                #mean_v2=mean_v2[mean_v2!=0]
                #print('v1 start:',datetime.datetime.fromtimestamp(mean_v1[0][0]/1000,timezone.utc),' | end:',datetime.datetime.fromtimestamp(mean_v1[-1][0]/1000,timezone.utc),'| len:',len(mean_v1))
                #print('v2 start:',datetime.datetime.fromtimestamp(mean_v2[0][0]/1000,timezone.utc),' | end:',datetime.datetime.fromtimestamp(mean_v2[-1][0]/1000,timezone.utc),'| len:',len(mean_v2))
                
                k=mean_v1/mean_v2
                print("v1:",mean_v1[:5],"| v2:",mean_v2[:5])
                k=k[k!=np.inf].mean()
                print('k not normaliz:',k,end='|')
                k=k
                
                print('k:',k)
                print('norm:',(mean_v2*k)[:10])
                
                # start_ku=ku.parse8601(start.isoformat())
                # end_ku=ku.parse8601(end.isoformat())

                # ku_can=self.parsku(start_ku,end_ku,timeframe)
                ku_can=ku_can_v[100:]
                print(f'len:{len(ku_can)}')
                try:
                    print(f'strt:{datetime.datetime.fromtimestamp(ku_can[0][0]/1000,datetime.timezone.utc)}')
                    print(f'end:{datetime.datetime.fromtimestamp(ku_can[-1][0]/1000,datetime.timezone.utc)}')
                except IndexError:
                    print(int((end-start).total_seconds()))
                    '''for kline in range(int((end-start).total_seconds()/)):
                        print(kline)'''
                #timestamps = [c[0] for c in m15]  # Получаем список всех timestamp
                ku_can_new=[]
                for inde,o in enumerate(ku_can):
                    if inde==0:
                        o[1]=data[-1][4]
                        o[2]=max(o[2],max(o[4],o[1]))
                        o[3]=min(o[3],min(o[4],o[1]))
                    if inde+1==len(ku_can):
                        o[4]=i[1]
                        o[2]=max(o[2],max(o[4],o[1]))
                        o[3]=min(o[3],min(o[4],o[1]))
                    o[5]=o[5]*k
                    ku_can_new.append(o)
                print(m15[-1][4])
                print(ku_can[0][1])
                print(ku_can[-1][4])
                print(i[1])
                print('last close:',m15[-1][4],' new open:',ku_can_new[0][1],'| new close:',ku_can_new[-1][4],' last open:',i[1])
                ku_can=ku_can_new
                dot.extend(ku_can)
                dot.append(i)

            #da['15m']['15m']
            #for candle in ku_can:
            #    if candle[0] not in timestamps:
            #        index = bisect.bisect_right(timestamps, candle[0])  # Найти правильное место
            #        m15.insert(index, candle)
            
            
                '''
                
                ge=requests.get('https://api.kucoin.com/api/v1/market/candles',params)
                ge.raise_for_status()
                print(f'strt:{datetime.datetime.fromtimestamp(ge[0][0])}')
                print(f'end:{datetime.datetime.fromtimestamp(ge[-1][6])}')
                #?symbol=BTC-USDT&type=1hour&startAt=1679616000&endAt=1679702400
                
                '''

            else:
                dot.append(i)
            last=datetime.datetime.fromtimestamp(i[0]/1000,datetime.timezone.utc)
        return dot


print(parse()(datetime.datetime.fromisoformat("2019-03-13 05:45:00+00:00")))


11
len m15:195
--------------------------
last0:2019-03-12 01:45:00+00:00
start1:2019-03-12T01:45:00+00:00
end1:2019-03-12T07:59:59+00:00
i:2019-03-12 08:00:00+00:00| last:2019-03-12 01:45:00+00:00
v1: [173.939279 138.345456 204.086555 151.221412 206.655249] | v2: [0.37046298 0.87657449 0.53838668 0.99408199 0.74258805]
k not normaliz: 546.9343343698202|k: 546.9343343698202
norm: [202.61892548 479.42868539 294.46215919 543.69756891 406.14690037
 825.48091009 448.28720084 623.79158602 399.15333556 130.93645761]
len:24
strt:2019-03-12 02:00:00+00:00
end:2019-03-12 07:45:00+00:00
3878.34
3878.34
3834.69
3834.69
last close: 3878.34  new open: 3878.34 | new close: 3834.69  last open: 3834.69
2:2019-03-12 00:00:00+00:00
2:2019-03-12 00:15:00+00:00
2:2019-03-12 00:30:00+00:00
2:2019-03-12 00:45:00+00:00
2:2019-03-12 01:00:00+00:00
2:2019-03-12 01:15:00+00:00
2:2019-03-12 01:30:00+00:00
2:2019-03-12 01:45:00+00:00
2:2019-03-12 02:00:00+00:00
2:2019-03-12 02:15:00+00:00
2:2019-03-12 02:30:00+00

In [5]:
a=np.array([1,2,3,4,5,6,7,8,9,9])
a[3:8]

array([4, 5, 6, 7, 8])

In [23]:
start=datetime.datetime.fromtimestamp(b['15m'][0][0]/1000,timezone.utc)
m15=b['15m']
print('start:',start)
#start1=datetime.datetime.strptime('2019-03-24 08:00:00+00:00',r'%Y-%m-%d %H:%M:%S%z')
#p=0
#while start1<start:
 #   start1=start1+datetime.timedelta(minutes=240)
  #  p+=1
#print(p)
#print(start1)
print("end:",datetime.datetime.fromtimestamp(m15[-1][0]/1000,timezone.utc))
for i,sv in enumerate(m15):
    tim=datetime.datetime.fromtimestamp(sv[0]/1000,timezone.utc)
    if tim!=start:
        print('----------------------------')
        print(f"error:{start} != {tim}")
        start=tim

    start=datetime.timedelta(minutes=15)+start
print(len(m15))

start: 2025-12-24 07:45:00+00:00
end: 2026-05-10 07:45:00+00:00
13153
